# RappiPay Fraud Detection Lab
## Construye un Cortex Agent con Cortex Code

**Autor**: Juan Camilo Villarreal | **Duracion**: 45 minutos

---

### Objetivo
Construir un agente AI conversacional (Cortex Agent) que responda preguntas sobre fraude de RappiPay en lenguaje natural.

### Que vas a aprender
1. Configurar un ambiente de datos de fraude en Snowflake
2. Usar **Cortex Code** para explorar datos y generar SQL
3. Crear una **Semantic View** para ensenarle a la AI tu modelo de datos
4. Crear un **Cortex Agent** y habilitarlo en **Snowflake Intelligence**

### Prerequisitos
- Cuenta Snowflake ([Solicita aqui](https://go.dataops.live/rappy-day/instructions))
- Cortex Code CLI instalado (ver instrucciones abajo)

### Instalar Cortex Code CLI

**macOS y Linux (incluyendo WSL):**
```bash
curl -LsS https://ai.snowflake.com/static/cc-scripts/install.sh | sh
```

**Windows (PowerShell):**
```powershell
irm https://ai.snowflake.com/static/cc-scripts/install.ps1 | iex
```

**Desktop App** (alternativa): [Descargar aqui](https://www.snowflake.com/en/product/snowflake-coco/downloads/)

Despues de instalar, verifica con:
```bash
cortex --version
```

### Contexto
Eres un Data Engineer en RappiPay. Tu equipo de fraude necesita consultar alertas y metricas sin escribir SQL. Vas a construir un asistente AI que responda sus preguntas en espanol.

> **Nota**: Los datos son ficticios pero inspirados en el mercado fintech colombiano/mexicano.

---
## Task 1: Setup del Ambiente (10 min)

**Objetivo**: Crear la base de datos, tablas y cargar datos sinteticos de fraude.

### Paso 1.1: Crear Database y Schemas

In [ ]:
USE ROLE ACCOUNTADMIN;

CREATE OR REPLACE DATABASE RAPPIPAY_DB;

CREATE OR REPLACE WAREHOUSE RAPPIPAY_WH
    WAREHOUSE_SIZE = 'XSMALL'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE
    INITIALLY_SUSPENDED = TRUE;

USE WAREHOUSE RAPPIPAY_WH;
USE DATABASE RAPPIPAY_DB;

CREATE OR REPLACE SCHEMA RAPPIPAY_DB.RAW;
CREATE OR REPLACE SCHEMA RAPPIPAY_DB.CURATED;
CREATE OR REPLACE SCHEMA RAPPIPAY_DB.ANALYTICS;
CREATE OR REPLACE SCHEMA RAPPIPAY_DB.APP;

### Paso 1.2: Crear Tablas

In [ ]:
USE SCHEMA RAPPIPAY_DB.RAW;

CREATE OR REPLACE TABLE RAPPIPAY_DB.RAW.TRANSACTIONS (
    transaction_id VARCHAR(36) NOT NULL,
    user_id VARCHAR(20) NOT NULL,
    merchant_id VARCHAR(20) NOT NULL,
    amount NUMBER(18,2) NOT NULL,
    currency VARCHAR(3) DEFAULT 'COP',
    transaction_type VARCHAR(30),
    channel VARCHAR(20),
    device_type VARCHAR(20),
    ip_address VARCHAR(45),
    location_city VARCHAR(50),
    location_country VARCHAR(3),
    status VARCHAR(20),
    created_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    description VARCHAR(500)
);

CREATE OR REPLACE TABLE RAPPIPAY_DB.RAW.USERS (
    user_id VARCHAR(20) NOT NULL,
    full_name VARCHAR(100) NOT NULL,
    email VARCHAR(150),
    phone VARCHAR(20),
    country VARCHAR(3),
    city VARCHAR(50),
    registration_date DATE,
    kyc_status VARCHAR(20),
    credit_score NUMBER(3),
    risk_level VARCHAR(10),
    verticals_used NUMBER(2),
    total_transactions NUMBER(10),
    account_status VARCHAR(20)
);

CREATE OR REPLACE TABLE RAPPIPAY_DB.RAW.MERCHANTS (
    merchant_id VARCHAR(20) NOT NULL,
    merchant_name VARCHAR(100) NOT NULL,
    category VARCHAR(50),
    city VARCHAR(50),
    country VARCHAR(3),
    risk_score NUMBER(5,2),
    avg_transaction_amount NUMBER(18,2),
    total_transactions NUMBER(10)
);

CREATE OR REPLACE TABLE RAPPIPAY_DB.RAW.FRAUD_ALERTS (
    alert_id VARCHAR(36) NOT NULL,
    transaction_id VARCHAR(36) NOT NULL,
    user_id VARCHAR(20) NOT NULL,
    alert_type VARCHAR(30),
    severity VARCHAR(10),
    status VARCHAR(30),
    created_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    resolved_at TIMESTAMP_NTZ,
    resolution_notes VARCHAR(1000),
    investigator_notes VARCHAR(2000)
);

### Paso 1.3: Cargar Datos Sinteticos (Merchants + Users)

In [ ]:
-- Merchants (120 registros)
INSERT INTO RAPPIPAY_DB.RAW.MERCHANTS (merchant_id, merchant_name, category, city, country, risk_score, avg_transaction_amount, total_transactions)
SELECT * FROM (
    SELECT 'MCH-' || LPAD(SEQ4()::VARCHAR, 4, '0') AS merchant_id,
           CASE MOD(SEQ4(), 40)
               WHEN 0 THEN 'Tienda D1' WHEN 1 THEN 'Exito' WHEN 2 THEN 'Oxxo'
               WHEN 3 THEN 'Rappi Restaurantes' WHEN 4 THEN 'Alkosto' WHEN 5 THEN 'Falabella'
               WHEN 6 THEN 'Transmilenio' WHEN 7 THEN 'Uber Colombia' WHEN 8 THEN 'DiDi Mexico'
               WHEN 9 THEN 'Cinepolis' WHEN 10 THEN 'Netflix CO' WHEN 11 THEN 'Spotify'
               WHEN 12 THEN 'Claro Pagos' WHEN 13 THEN 'ETB Telecomunicaciones' WHEN 14 THEN 'EPM Servicios'
               WHEN 15 THEN 'Bancolombia Transferencia' WHEN 16 THEN 'Nequi Transfer' WHEN 17 THEN 'Daviplata Envio'
               WHEN 18 THEN 'Jumbo Supermercado' WHEN 19 THEN 'Carulla Fresh' WHEN 20 THEN 'Soriana'
               WHEN 21 THEN 'Liverpool' WHEN 22 THEN 'Mercado Libre CO' WHEN 23 THEN 'Amazon MX'
               WHEN 24 THEN 'Rappi Travel' WHEN 25 THEN 'Avianca' WHEN 26 THEN 'Volaris'
               WHEN 27 THEN 'Bodytech Gym' WHEN 28 THEN 'Smart Fit' WHEN 29 THEN 'Farmatodo'
               WHEN 30 THEN 'Drogueria Olimpica' WHEN 31 THEN 'Homecenter' WHEN 32 THEN 'Coppel'
               WHEN 33 THEN 'Wom Telecomunicaciones' WHEN 34 THEN 'Tigo Pagos' WHEN 35 THEN 'Movistar Recarga'
               WHEN 36 THEN 'Crepes & Waffles' WHEN 37 THEN 'Frisby' WHEN 38 THEN 'El Corral'
               WHEN 39 THEN 'Dominos Pizza'
           END AS merchant_name,
           CASE MOD(SEQ4(), 6)
               WHEN 0 THEN 'supermercado' WHEN 1 THEN 'restaurante' WHEN 2 THEN 'transporte'
               WHEN 3 THEN 'entretenimiento' WHEN 4 THEN 'pagos_servicios' WHEN 5 THEN 'transferencia'
           END AS category,
           CASE MOD(SEQ4(), 10)
               WHEN 0 THEN 'Bogota' WHEN 1 THEN 'Medellin' WHEN 2 THEN 'Cali'
               WHEN 3 THEN 'Barranquilla' WHEN 4 THEN 'Cartagena' WHEN 5 THEN 'CDMX'
               WHEN 6 THEN 'Guadalajara' WHEN 7 THEN 'Monterrey' WHEN 8 THEN 'Bucaramanga'
               WHEN 9 THEN 'Pereira'
           END AS city,
           CASE WHEN MOD(SEQ4(), 10) >= 5 THEN 'MEX' ELSE 'COL' END AS country,
           ROUND(UNIFORM(10, 95, RANDOM())::NUMBER(5,2), 2) AS risk_score,
           ROUND(UNIFORM(15000, 800000, RANDOM())::NUMBER(18,2), 2) AS avg_transaction_amount,
           UNIFORM(100, 50000, RANDOM()) AS total_transactions
    FROM TABLE(GENERATOR(ROWCOUNT => 120))
);

-- Users (600 registros)
INSERT INTO RAPPIPAY_DB.RAW.USERS (user_id, full_name, email, phone, country, city, registration_date, kyc_status, credit_score, risk_level, verticals_used, total_transactions, account_status)
SELECT * FROM (
    SELECT 'USR-' || LPAD(SEQ4()::VARCHAR, 5, '0') AS user_id,
           CASE MOD(SEQ4(), 30)
               WHEN 0 THEN 'Carlos Andres Martinez' WHEN 1 THEN 'Maria Fernanda Lopez'
               WHEN 2 THEN 'Juan Pablo Hernandez' WHEN 3 THEN 'Laura Valentina Garcia'
               WHEN 4 THEN 'Santiago Ramirez Ortiz' WHEN 5 THEN 'Camila Andrea Torres'
               WHEN 6 THEN 'Andres Felipe Morales' WHEN 7 THEN 'Daniela Alejandra Ruiz'
               WHEN 8 THEN 'Diego Armando Perez' WHEN 9 THEN 'Valentina Sofia Castro'
               WHEN 10 THEN 'Sebastian David Diaz' WHEN 11 THEN 'Isabella Restrepo Mejia'
               WHEN 12 THEN 'Miguel Angel Rodriguez' WHEN 13 THEN 'Natalia Andrea Vargas'
               WHEN 14 THEN 'Alejandro Jose Mendoza' WHEN 15 THEN 'Paula Andrea Gutierrez'
               WHEN 16 THEN 'Fernando Enrique Salazar' WHEN 17 THEN 'Ana Maria Cardenas'
               WHEN 18 THEN 'Roberto Carlos Aguilar' WHEN 19 THEN 'Monica Patricia Rios'
               WHEN 20 THEN 'Oscar Eduardo Pineda' WHEN 21 THEN 'Claudia Milena Bernal'
               WHEN 22 THEN 'Jorge Luis Camacho' WHEN 23 THEN 'Sandra Liliana Parra'
               WHEN 24 THEN 'Pedro Antonio Rojas' WHEN 25 THEN 'Carolina Gomez Uribe'
               WHEN 26 THEN 'Luis Fernando Ochoa' WHEN 27 THEN 'Andrea del Pilar Suarez'
               WHEN 28 THEN 'Cristian Camilo Duarte' WHEN 29 THEN 'Diana Marcela Quintero'
           END AS full_name,
           'user_' || SEQ4() || '@rappi.com' AS email,
           '+57' || LPAD(UNIFORM(3001000000, 3209999999, RANDOM())::VARCHAR, 10, '0') AS phone,
           CASE WHEN MOD(SEQ4(), 5) = 0 THEN 'MEX' ELSE 'COL' END AS country,
           CASE MOD(SEQ4(), 10)
               WHEN 0 THEN 'Bogota' WHEN 1 THEN 'Medellin' WHEN 2 THEN 'Cali'
               WHEN 3 THEN 'Barranquilla' WHEN 4 THEN 'Cartagena' WHEN 5 THEN 'CDMX'
               WHEN 6 THEN 'Guadalajara' WHEN 7 THEN 'Monterrey' WHEN 8 THEN 'Bucaramanga'
               WHEN 9 THEN 'Pereira'
           END AS city,
           DATEADD(DAY, -UNIFORM(30, 1095, RANDOM()), CURRENT_DATE()) AS registration_date,
           CASE MOD(SEQ4(), 4) WHEN 0 THEN 'verified' WHEN 1 THEN 'verified' WHEN 2 THEN 'pending' WHEN 3 THEN 'under_review' END AS kyc_status,
           UNIFORM(300, 850, RANDOM()) AS credit_score,
           CASE WHEN UNIFORM(1, 100, RANDOM()) <= 10 THEN 'high'
                WHEN UNIFORM(1, 100, RANDOM()) <= 35 THEN 'medium'
                ELSE 'low' END AS risk_level,
           UNIFORM(1, 5, RANDOM()) AS verticals_used,
           UNIFORM(5, 2000, RANDOM()) AS total_transactions,
           CASE WHEN UNIFORM(1, 100, RANDOM()) <= 5 THEN 'suspended'
                WHEN UNIFORM(1, 100, RANDOM()) <= 10 THEN 'under_review'
                ELSE 'active' END AS account_status
    FROM TABLE(GENERATOR(ROWCOUNT => 600))
);

### Paso 1.4: Cargar Transacciones y Alertas de Fraude

In [ ]:
-- Transactions (12000 registros)
INSERT INTO RAPPIPAY_DB.RAW.TRANSACTIONS (transaction_id, user_id, merchant_id, amount, currency, transaction_type, channel, device_type, ip_address, location_city, location_country, status, created_at, description)
SELECT * FROM (
    SELECT UUID_STRING() AS transaction_id,
           'USR-' || LPAD(UNIFORM(0, 599, RANDOM())::VARCHAR, 5, '0') AS user_id,
           'MCH-' || LPAD(UNIFORM(0, 119, RANDOM())::VARCHAR, 4, '0') AS merchant_id,
           ROUND(UNIFORM(5000, 5000000, RANDOM())::NUMBER(18,2), 2) AS amount,
           CASE WHEN UNIFORM(1, 5, RANDOM()) = 1 THEN 'MXN' ELSE 'COP' END AS currency,
           CASE MOD(SEQ4(), 6)
               WHEN 0 THEN 'purchase' WHEN 1 THEN 'transfer' WHEN 2 THEN 'withdrawal'
               WHEN 3 THEN 'payment' WHEN 4 THEN 'refund' WHEN 5 THEN 'top_up'
           END AS transaction_type,
           CASE MOD(SEQ4(), 4)
               WHEN 0 THEN 'app_mobile' WHEN 1 THEN 'web' WHEN 2 THEN 'pos_terminal' WHEN 3 THEN 'qr_code'
           END AS channel,
           CASE MOD(SEQ4(), 4)
               WHEN 0 THEN 'android' WHEN 1 THEN 'ios' WHEN 2 THEN 'desktop' WHEN 3 THEN 'tablet'
           END AS device_type,
           CONCAT(UNIFORM(10, 223, RANDOM())::VARCHAR, '.', UNIFORM(0, 255, RANDOM())::VARCHAR, '.', UNIFORM(0, 255, RANDOM())::VARCHAR, '.', UNIFORM(1, 254, RANDOM())::VARCHAR) AS ip_address,
           CASE MOD(SEQ4(), 10)
               WHEN 0 THEN 'Bogota' WHEN 1 THEN 'Medellin' WHEN 2 THEN 'Cali'
               WHEN 3 THEN 'Barranquilla' WHEN 4 THEN 'Cartagena' WHEN 5 THEN 'CDMX'
               WHEN 6 THEN 'Guadalajara' WHEN 7 THEN 'Monterrey' WHEN 8 THEN 'Bucaramanga'
               WHEN 9 THEN 'Pereira'
           END AS location_city,
           CASE WHEN MOD(SEQ4(), 10) >= 5 THEN 'MEX' ELSE 'COL' END AS location_country,
           CASE WHEN UNIFORM(1, 100, RANDOM()) <= 3 THEN 'declined'
                WHEN UNIFORM(1, 100, RANDOM()) <= 8 THEN 'flagged'
                WHEN UNIFORM(1, 100, RANDOM()) <= 12 THEN 'pending_review'
                ELSE 'approved' END AS status,
           DATEADD(MINUTE, -UNIFORM(1, 525600, RANDOM()), CURRENT_TIMESTAMP()) AS created_at,
           CASE MOD(SEQ4(), 12)
               WHEN 0 THEN 'Compra en supermercado productos basicos'
               WHEN 1 THEN 'Pago de servicio de transporte urbano'
               WHEN 2 THEN 'Transferencia a cuenta de tercero'
               WHEN 3 THEN 'Pago de factura de servicios publicos'
               WHEN 4 THEN 'Compra en restaurante domicilio'
               WHEN 5 THEN 'Recarga de celular prepago'
               WHEN 6 THEN 'Pago de suscripcion streaming'
               WHEN 7 THEN 'Compra en linea marketplace'
               WHEN 8 THEN 'Retiro de efectivo cajero automatico'
               WHEN 9 THEN 'Pago de cuota de credito'
               WHEN 10 THEN 'Transferencia internacional remesa'
               WHEN 11 THEN 'Compra tiquete de vuelo nacional'
           END AS description
    FROM TABLE(GENERATOR(ROWCOUNT => 12000))
);

-- Fraud Alerts (250 registros)
INSERT INTO RAPPIPAY_DB.RAW.FRAUD_ALERTS (alert_id, transaction_id, user_id, alert_type, severity, status, created_at, resolved_at, resolution_notes, investigator_notes)
SELECT * FROM (
    SELECT UUID_STRING() AS alert_id, t.transaction_id, t.user_id,
           CASE MOD(SEQ4(), 5) WHEN 0 THEN 'identity_theft' WHEN 1 THEN 'card_cloning' WHEN 2 THEN 'account_takeover' WHEN 3 THEN 'phishing' WHEN 4 THEN 'money_laundering' END AS alert_type,
           CASE MOD(SEQ4(), 3) WHEN 0 THEN 'critical' WHEN 1 THEN 'high' WHEN 2 THEN 'medium' END AS severity,
           CASE MOD(SEQ4(), 4) WHEN 0 THEN 'open' WHEN 1 THEN 'investigating' WHEN 2 THEN 'resolved_fraud' WHEN 3 THEN 'false_positive' END AS status,
           DATEADD(MINUTE, -UNIFORM(1, 43200, RANDOM()), CURRENT_TIMESTAMP()) AS created_at,
           CASE WHEN MOD(SEQ4(), 4) >= 2 THEN DATEADD(MINUTE, -UNIFORM(1, 10080, RANDOM()), CURRENT_TIMESTAMP()) ELSE NULL END AS resolved_at,
           CASE MOD(SEQ4(), 5)
               WHEN 0 THEN 'Confirmado como fraude. Se bloqueo tarjeta y se inicio reembolso.'
               WHEN 1 THEN 'Falso positivo. Usuario confirmo la transaccion.'
               WHEN 2 THEN 'Fraude confirmado por clonacion de tarjeta.'
               WHEN 3 THEN 'Caso escalado a autoridades por patron de lavado.'
               WHEN 4 THEN 'Usuario victima de phishing. Credenciales restablecidas.'
           END AS resolution_notes,
           CASE MOD(SEQ4(), 6)
               WHEN 0 THEN 'Transaccion desde IP desconocida en horario inusual. Dispositivo nuevo. Monto 4x superior al promedio.'
               WHEN 1 THEN 'Compras rapidas en multiples merchants en menos de 5 minutos. Geolocalizacion inconsistente.'
               WHEN 2 THEN 'Cambio de contrasena seguido de transferencia al limite maximo. IP de VPN.'
               WHEN 3 THEN 'Microtransacciones hacia misma cuenta en intervalos de 2 min. Patron de structuring.'
               WHEN 4 THEN 'Usuario no reconoce transaccion. Ultima sesion hace 72 horas. Posible credential stuffing.'
               WHEN 5 THEN 'Multiples intentos fallidos de auth seguidos de transaccion exitosa. Posible brute force.'
           END AS investigator_notes
    FROM (SELECT transaction_id, user_id, ROW_NUMBER() OVER (ORDER BY RANDOM()) AS rn
          FROM RAPPIPAY_DB.RAW.TRANSACTIONS WHERE status IN ('flagged', 'declined', 'pending_review') QUALIFY rn <= 250) t
);

### Paso 1.5: Validar Datos

In [ ]:
-- Validar que todo se cargo correctamente
SELECT 'TRANSACTIONS' AS tabla, COUNT(*) AS filas FROM RAPPIPAY_DB.RAW.TRANSACTIONS
UNION ALL SELECT 'USERS', COUNT(*) FROM RAPPIPAY_DB.RAW.USERS
UNION ALL SELECT 'MERCHANTS', COUNT(*) FROM RAPPIPAY_DB.RAW.MERCHANTS
UNION ALL SELECT 'FRAUD_ALERTS', COUNT(*) FROM RAPPIPAY_DB.RAW.FRAUD_ALERTS;

### Checklist Task 1
- [ ] RAPPIPAY_DB creada con 4 schemas
- [ ] 12,000 transacciones, 600 usuarios, 120 merchants, 250 alertas
- [ ] Datos realistas (Bogota, Medellin, Oxxo, Nequi, etc.)

---

## Task 2: Explorar Datos con Cortex Code (10 min)

**Objetivo**: Usar Cortex Code para descubrir patrones de fraude en los datos.

### Paso 2.1: Analizar el modelo de datos

Abre **Cortex Code** y copia este prompt:

```
Conectate a RAPPIPAY_DB y analiza las tablas en el schema RAW.
Dame un resumen: que tablas hay, cuantos registros, como se relacionan,
y que campos son relevantes para deteccion de fraude.
```

### Paso 2.2: Descubrir patrones de fraude

```
Usando RAPPIPAY_DB.RAW, analiza patrones de fraude:
1. Tasa de fraude general (flagged+declined vs total)
2. Tipos de fraude mas comunes en FRAUD_ALERTS
3. Merchants con mayor tasa de transacciones sospechosas
4. Correlacion entre monto y probabilidad de fraude
Muestra queries SQL ejecutables con resultados.
```

### Paso 2.3: Ejecuta este analisis para comparar

In [ ]:
SELECT 
    COUNT(*) AS total_transacciones,
    COUNT(CASE WHEN status IN ('flagged', 'declined') THEN 1 END) AS sospechosas,
    ROUND(COUNT(CASE WHEN status IN ('flagged', 'declined') THEN 1 END) * 100.0 / COUNT(*), 2) AS tasa_fraude_pct,
    ROUND(SUM(CASE WHEN status IN ('flagged', 'declined') THEN amount ELSE 0 END), 0) AS monto_en_riesgo_cop
FROM RAPPIPAY_DB.RAW.TRANSACTIONS;

In [ ]:
SELECT alert_type, severity, COUNT(*) AS total,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS porcentaje
FROM RAPPIPAY_DB.RAW.FRAUD_ALERTS
GROUP BY 1, 2
ORDER BY 3 DESC;

### Checklist Task 2
- [ ] Cortex Code analizo el modelo de datos correctamente
- [ ] Identificaste la tasa de fraude y patrones principales
- [ ] Sabes cuales merchants son de mayor riesgo

---

## Task 3: Crear Semantic View (10 min)

**Objetivo**: Crear una Semantic View que ensene a Cortex Analyst tu modelo de datos.

### Que es una Semantic View?
Un objeto SQL que define significado: metricas, dimensiones, relaciones y sinonimos. Es lo que permite preguntas en lenguaje natural.

### Paso 3.1: Generar con Cortex Code

```
Crea una Semantic View SQL para RAPPIPAY_DB que permita preguntas sobre fraude.
Basate en RAPPIPAY_DB.RAW.TRANSACTIONS, USERS, MERCHANTS y FRAUD_ALERTS.
Define metricas: total_transacciones, monto_total, tasa_fraude, alertas_activas.
Dimensiones: ciudad, pais, tipo_transaccion, canal, merchant, severidad, alert_type.
Sinonimos en espanol sin acentos. Usa CREATE OR REPLACE SEMANTIC VIEW.
Ejecuta el SQL.
```

### Paso 3.2: Verificar

In [ ]:
-- Verificar semantic views existentes
SHOW SEMANTIC VIEWS IN SCHEMA RAPPIPAY_DB.ANALYTICS;

### Checklist Task 3
- [ ] Semantic View creada
- [ ] SHOW SEMANTIC VIEWS muestra el objeto

---

## Task 4: Crear Cortex Agent + Intelligence (10 min)

**Objetivo**: Crear un agente que responda preguntas de fraude en espanol.

### Paso 4.1: Crear el Agent con Cortex Code

```
Crea un Cortex Agent en RAPPIPAY_DB.APP llamado RAPPIPAY_FRAUD_ANALYST.
Debe usar la semantic view que creamos. Instrucciones del agente:
"Eres un analista de fraude de RappiPay. Responde sobre transacciones 
sospechosas, alertas y metricas de riesgo. Incluye numeros concretos.
Responde en espanol."
Preguntas de ejemplo:
- Cuantas alertas de fraude estan abiertas?
- Cual es la tasa de fraude por ciudad?
- Que merchants tienen mas transacciones sospechosas?
Otorga USAGE a PUBLIC. Ejecuta todo.
```

### Paso 4.2: SQL alternativo

In [ ]:
-- Crear el Cortex Agent
CREATE OR REPLACE CORTEX AGENT RAPPIPAY_DB.APP.RAPPIPAY_FRAUD_ANALYST
  COMMENT = 'Agente de analisis de fraude para RappiPay'
  SYSTEM_PROMPT = 'Eres un analista de fraude de RappiPay. Responde preguntas sobre transacciones sospechosas, alertas de fraude, y metricas de riesgo. Siempre incluye numeros concretos. Responde en espanol.'
  SAMPLE_QUESTIONS = (
    'Cuantas alertas de fraude estan abiertas?',
    'Cual es la tasa de fraude por ciudad?',
    'Que merchants tienen mas transacciones sospechosas?',
    'Cual es el monto total en riesgo?',
    'Que tipo de fraude es mas comun?'
  );

### Paso 4.3: Habilitar en Snowflake Intelligence

In [ ]:
-- Grants para acceso
GRANT USAGE ON DATABASE RAPPIPAY_DB TO ROLE PUBLIC;
GRANT USAGE ON SCHEMA RAPPIPAY_DB.APP TO ROLE PUBLIC;
GRANT USAGE ON CORTEX AGENT RAPPIPAY_DB.APP.RAPPIPAY_FRAUD_ANALYST TO ROLE PUBLIC;

-- Verificar
SHOW CORTEX AGENTS IN SCHEMA RAPPIPAY_DB.APP;

### Paso 4.4: Probar el agente

Ve a **Snowsight > Snowflake Intelligence** y prueba:

1. "Cuantas alertas de fraude estan abiertas?"
2. "Cual es la tasa de fraude por ciudad?"
3. "Que merchants tienen mas transacciones sospechosas?"
4. "Compara fraude entre Colombia y Mexico"
5. "Dame un resumen ejecutivo del estado del fraude"

### Checklist Task 4
- [ ] Agent RAPPIPAY_FRAUD_ANALYST creado
- [ ] SHOW CORTEX AGENTS lo muestra
- [ ] Funciona en Snowflake Intelligence
- [ ] Responde en espanol con datos correctos

---

## Task 5: Validacion Final (5 min)

### Preguntas avanzadas para tu presentacion

| Pregunta | Demuestra |
|----------|-----------|
| "Dame un resumen ejecutivo del fraude este mes" | Sintesis AI |
| "Que patron de fraude crece mas rapido?" | Tendencias |
| "Compara riesgo entre supermercados y transporte" | Segmentacion |
| "Top 5 usuarios con mas alertas" | Investigacion |

### Cleanup (cuando termines)

In [ ]:
-- RESET: Descomenta para limpiar todo
-- DROP DATABASE IF EXISTS RAPPIPAY_DB;
-- DROP WAREHOUSE IF EXISTS RAPPIPAY_WH;

---
## Resumen

| Componente | Producto | Funcion |
|-----------|---------|---------|
| Datos sinteticos | Tables | 12K transacciones de fraude fintech |
| Semantic View | Cortex Analyst | Modelo de datos para lenguaje natural |
| Cortex Agent | Intelligence | Chat sobre fraude en espanol |
| Exploracion | Cortex Code | SQL y analisis asistido por AI |

---
*Lab creado por Juan Camilo Villarreal | Snowflake SE LATAM*